 AdaBoost weights; XGBoost early stopping; LightGBM timing; SHAP beeswarm; v1 vs v2 bar chart; boosting_benchmark() function

In [ ]:
import time
import lightgbm as lgb
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 1. Setup Dataset & Preprocessor
np.random.seed(42)
n_samples = 300

data = {
    'Branch': np.random.choice(['CSE', 'ECE', 'MECH', 'CIVIL'], n_samples),
    'Gender': np.random.choice(['Male', 'Female'], n_samples),
    'Years_Experience': np.random.uniform(0, 3, n_samples),
    'Internships': np.random.randint(0, 4, n_samples),
    'PlacementStatus': np.random.choice([0, 1], n_samples, p=[0.4, 0.6]),
}
df = pd.DataFrame(data)

X = df.drop(columns=['PlacementStatus'])
y_binary = df['PlacementStatus'].astype(int)

categorical_cols = X.select_dtypes(
    include=['object', 'category']
).columns.tolist()
numerical_cols = X.select_dtypes(
    include=['int64', 'float64', 'int32', 'float32']
).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        (
            'cat',
            OneHotEncoder(handle_unknown='ignore', drop='first'),
            categorical_cols,
        ),
    ]
)

X_train, X_val, y_train, y_val = train_test_split(
    X, y_binary, test_size=0.2, random_state=42
)

# Transform features
preprocessed_X_train = preprocessor.fit_transform(X_train)
preprocessed_X_val = preprocessor.transform(X_val)


# 2. Benchmark Function Definition
def boosting_benchmark(X_tr, y_tr, X_v, y_v):
  """Benchmarks AdaBoost, XGBoost, and LightGBM models."""
  results = []

  # AdaBoost
  start_time = time.time()
  ada = AdaBoostClassifier(n_estimators=100, random_state=42)
  ada.fit(X_tr, y_tr)
  ada_time = time.time() - start_time
  ada_acc = accuracy_score(y_v, ada.predict(X_v))
  results.append({
      'Model': 'AdaBoost',
      'Val_Accuracy': ada_acc,
      'Train_Time_sec': ada_time,
  })

  # XGBoost with Early Stopping
  start_time = time.time()
  xgb_clf = xgb.XGBClassifier(
      n_estimators=200,
      learning_rate=0.1,
      early_stopping_rounds=10,
      random_state=42,
  )
  xgb_clf.fit(X_tr, y_tr, eval_set=[(X_v, y_v)], verbose=False)
  xgb_time = time.time() - start_time
  xgb_acc = accuracy_score(y_v, xgb_clf.predict(X_v))
  results.append({
      'Model': 'XGBoost',
      'Val_Accuracy': xgb_acc,
      'Train_Time_sec': xgb_time,
  })

  # LightGBM
  start_time = time.time()
  lgb_clf = lgb.LGBMClassifier(
      n_estimators=100, learning_rate=0.1, random_state=42, verbose=-1
  )
  lgb_clf.fit(X_tr, y_tr)
  lgb_time = time.time() - start_time
  lgb_acc = accuracy_score(y_v, lgb_clf.predict(X_v))
  results.append({
      'Model': 'LightGBM',
      'Val_Accuracy': lgb_acc,
      'Train_Time_sec': lgb_time,
  })

  return pd.DataFrame(results)


# 3. Run Benchmark
benchmark_df = boosting_benchmark(
    preprocessed_X_train, y_train, preprocessed_X_val, y_val
)
print('=== Boosting Benchmark Results ===')
print(benchmark_df)

=== Boosting Benchmark Results ===
      Model  Val_Accuracy  Train_Time_sec
0  AdaBoost      0.633333        0.471382
1   XGBoost      0.583333        0.161965
2  LightGBM      0.550000        0.125191


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


 Write boosting_benchmark(X_train, y_train, X_val, y_val) — fits AdaBoost, GBC, XGB, LGB; returns DataFrame sorted by val_accuracy
descending

In [ ]:
import lightgbm as lgb
import pandas as pd
import xgboost as xgb
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score


def boosting_benchmark(X_train, y_train, X_val, y_val):
  """Fits AdaBoost, Gradient Boosting (GBC), XGBoost, and LightGBM.

  Returns a DataFrame sorted by validation accuracy in descending order.
  """
  results = []

  # 1. AdaBoost
  ada = AdaBoostClassifier(random_state=42)
  ada.fit(X_train, y_train)
  ada_acc = accuracy_score(y_val, ada.predict(X_val))
  results.append({'Model': 'AdaBoost', 'val_accuracy': ada_acc})

  # 2. Gradient Boosting Classifier (GBC)
  gbc = GradientBoostingClassifier(random_state=42)
  gbc.fit(X_train, y_train)
  gbc_acc = accuracy_score(y_val, gbc.predict(X_val))
  results.append({'Model': 'GradientBoosting (GBC)', 'val_accuracy': gbc_acc})

  # 3. XGBoost
  xgb_clf = xgb.XGBClassifier(random_state=42)
  xgb_clf.fit(X_train, y_train)
  xgb_acc = accuracy_score(y_val, xgb_clf.predict(X_val))
  results.append({'Model': 'XGBoost', 'val_accuracy': xgb_acc})

  # 4. LightGBM
  lgb_clf = lgb.LGBMClassifier(random_state=42, verbose=-1)
  lgb_clf.fit(X_train, y_train)
  lgb_acc = accuracy_score(y_val, lgb_clf.predict(X_val))
  results.append({'Model': 'LightGBM', 'val_accuracy': lgb_acc})

  # Convert to DataFrame and sort by val_accuracy descending
  df_results = pd.DataFrame(results)
  df_results = df_results.sort_values(
      by='val_accuracy', ascending=False
  ).reset_index(drop=True)

  return df_results


# --- Example Execution ---
benchmark_df = boosting_benchmark(
    preprocessed_X_train, y_train, preprocessed_X_val, y_val
)
print(benchmark_df)

                    Model  val_accuracy
0                AdaBoost      0.600000
1  GradientBoosting (GBC)      0.550000
2                LightGBM      0.550000
3                 XGBoost      0.533333


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
